Q1. What is Gradient Boosting Regression?

Gradient Boosting Regression (GBR) is an ensemble technique for regression that builds an ensemble of weak learners (usually shallow decision trees) sequentially.

Each tree corrects the errors (residuals) of the previous trees using gradient descent on a loss function.

Goal: Minimize a differentiable loss function, typically Mean Squared Error (MSE) for regression.

Key properties:

Sequential learning → each new tree focuses on residual errors.

Combines predictions of all trees to produce a strong model.

Flexible → can optimize various loss functions.

Q2. Implement a simple Gradient Boosting from scratch

We’ll implement a very simplified version using decision stumps (1-level trees) for illustration.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Simple dataset
X = np.array([[1], [2], [3], [4], [5], [6]]).astype(float)
y = np.array([1.5, 1.7, 3.2, 3.8, 5.0, 6.1])

# Hyperparameters
n_estimators = 5
learning_rate = 0.1
max_depth = 1  # weak learner

# Initialize predictions
y_pred = np.zeros(y.shape)
trees = []

for i in range(n_estimators):
    # Compute residuals
    residuals = y - y_pred

    # Fit a weak learner on residuals
    tree = DecisionTreeRegressor(max_depth=max_depth)
    tree.fit(X, residuals)
    trees.append(tree)

    # Update predictions
    update = learning_rate * tree.predict(X)
    y_pred += update

    print(f"Iteration {i+1}, Residuals: {residuals}, Update: {update}")

# Evaluate performance
mse = mean_squared_error(y, y_pred)
r2 = r2_score(y, y_pred)
print(f"\nFinal predictions: {y_pred}")
print(f"MSE: {mse:.4f}, R2: {r2:.4f}")

Explanation:

Start with zero predictions.

Compute residuals = difference between true y and predicted y_pred.

Fit a weak learner (decision stump) on the residuals.

Update predictions with a learning rate scaled contribution from the new tree.

Repeat for n_estimators.

Final predictions = sum of all trees’ contributions.

Q3. Experiment with hyperparameters

Hyperparameters to tune:

n_estimators → number of trees

learning_rate → how much each tree contributes

max_depth → complexity of each tree

Example using grid search manually:

In [ ]:
best_mse = float('inf')
best_params = {}

for n in [5, 10, 20]:
    for lr in [0.05, 0.1, 0.2]:
        for depth in [1, 2]:
            y_pred = np.zeros(y.shape)
            for i in range(n):
                residuals = y - y_pred
                tree = DecisionTreeRegressor(max_depth=depth)
                tree.fit(X, residuals)
                y_pred += lr * tree.predict(X)
            mse = mean_squared_error(y, y_pred)
            if mse < best_mse:
                best_mse = mse
                best_params = {'n_estimators': n, 'learning_rate': lr, 'max_depth': depth}

print(f"Best parameters: {best_params}, MSE: {best_mse:.4f}")

For larger datasets, use scikit-learn’s GridSearchCV or RandomizedSearchCV with GradientBoostingRegressor.

Q4. What is a weak learner in Gradient Boosting?

A weak learner is a model that is slightly better than random guessing.

In GBR, weak learners are usually shallow decision trees (stumps).

Each weak learner corrects the residuals of previous learners.

Q5. Intuition behind Gradient Boosting

Start with a simple model.

Calculate errors (residuals) of this model.

Fit a new model to predict the residuals.

Add this model to the ensemble.

Repeat → each new model focuses on correcting mistakes of the previous ones.

Analogy: “Take small steps in the direction that reduces the error the most” → gradient descent in function space.

Q6. How Gradient Boosting builds an ensemble of weak learners

Fit first weak learner
ℎ
1
(
𝑥
)
h
1
	​

(x) to predict
𝑦
y.

Compute residuals:
𝑟
𝑖
=
𝑦
𝑖
−
ℎ
1
(
𝑥
𝑖
)
r
i
	​

=y
i
	​

−h
1
	​

(x
i
	​

)

Fit second weak learner
ℎ
2
(
𝑥
)
h
2
	​

(x) to predict
𝑟
𝑖
r
i
	​

.

Update predictions:
𝑦
^
=
ℎ
1
(
𝑥
)
+
𝜂
ℎ
2
(
𝑥
)
y
^
	​

=h
1
	​

(x)+ηh
2
	​

(x)

Repeat for n_estimators weak learners.

Final ensemble prediction: sum of all learners scaled by learning rate.

Q7. Steps for constructing mathematical intuition

Start with a loss function
𝐿
(
𝑦
,
𝐹
(
𝑥
)
)
L(y,F(x)) (e.g., MSE).

Initialize model with constant value minimizing loss:
𝐹
0
(
𝑥
)
=
arg
⁡
min
⁡
𝛾
∑
𝑖
𝐿
(
𝑦
𝑖
,
𝛾
)
F
0
	​

(x)=argmin
γ
	​

∑
i
	​

L(y
i
	​

,γ)

Compute pseudo-residuals:

𝑟
𝑖
𝑚
=
−
[
∂
𝐿
(
𝑦
𝑖
,
𝐹
(
𝑥
𝑖
)
)
∂
𝐹
(
𝑥
𝑖
)
]
𝐹
(
𝑥
)
=
𝐹
𝑚
−
1
(
𝑥
)
r
im
	​

=−[
∂F(x
i
	​

)
∂L(y
i
	​

,F(x
i
	​

))
	​

]
F(x)=F
m−1
	​

(x)
	​


Fit weak learner
ℎ
𝑚
(
𝑥
)
h
m
	​

(x) to pseudo-residuals.

Compute multiplier
𝛾
𝑚
γ
m
	​

 to minimize loss along this direction.

Update model:

𝐹
𝑚
(
𝑥
)
=
𝐹
𝑚
−
1
(
𝑥
)
+
𝜂
𝛾
𝑚
ℎ
𝑚
(
𝑥
)
F
m
	​

(x)=F
m−1
	​

(x)+ηγ
m
	​

h
m
	​

(x)

Repeat steps 3–6 until convergence or n_estimators reached.

Essentially, Gradient Boosting performs gradient descent in function space, building an ensemble that iteratively reduces the loss.